# Строгое вычисление 30 000 знаков постоянной Голомба–Дикмана

$$\lambda=\int_0^\infty F(u)\,du,\qquad F(u)=e^{-u-E_1(u)}=0.62432998854355087099\ldots$$

**Что доказывает этот ноутбук.** Для целых $D=30000$ и $G=30$ он строит целочисленный интервал $[\Lambda_-,\Lambda_+]\ni\lambda$ и проверяет, что $\lfloor 10^D\Lambda_-\rfloor=\lfloor 10^D\Lambda_+\rfloor$. Тогда первые $D$ десятичных знаков $\lambda$ **доказаны** (усечение, без округления).

Интервал получается из **явной суммы** по правилу трапеций с подстановкой Такахаси–Мори плюс **доказанные оценки** всех остальных ошибок (хвосты суммы, дискретизация, остатки рядов). Ничего не оценивается «на глаз»; двойная точность (`float`) используется только для *выбора* метода и точности, но не входит ни в одну границу.

**На чём держится доказательство (база доверия):**
1. корректность интервальной арифметики Arb (`python-flint`): каждый `arb` — шар, содержащий истинное значение;
2. теорема о трапециях для аналитических функций в полосе (Trefethen–Weideman, *SIAM Review* 56 (2014), Thm 5.1) — цитируется;
3. классическая теорема Стилтьеса о двусторонних оценках подходящими дробями (Лемма 7; цитируется: Wall, *Analytic Theory of Continued Fractions*; Jones–Thron);
4. леммы §1 ниже (их доказательства приведены);
5. целочисленная арифметика Python (точная).

`selftest` ниже проверяет только *семантику методов* `python-flint`, на которые опирается код (точные `lower/upper`, `floor().unique_fmpz()` и т.д.), но не заменяет доверие к Arb.

## §1. Математика

Обозначения: $E_1(u)=\int_u^\infty e^{-s}s^{-1}ds$, $\gamma$ — постоянная Эйлера,
$$u(w)=\exp\!\big(w-e^{-w}\big),\qquad G(w)=F(u(w))\,u(w)\,\big(1+e^{-w}\big)=F(u(w))\,u'(w),\quad w\in\mathbb C.$$

**Лемма 1 (формулы для $F$).**
(a) $F(u)=e^{\gamma}\,u\,\exp\!\big(-2u+\sum_{k\ge2}\tfrac{(-u)^k}{k\,k!}\big)$ — целая функция.
(b) При $u>0$: $0<F(u)<\min(e^\gamma u,\ e^{-u})$.
(c) При $u>0$, $K\ge0$: $\;e^{u}E_1(u)=\sum_{j<K}\tfrac{(-1)^j j!}{u^{j+1}}+R_K$, $\;|R_K|\le \tfrac{K!}{u^{K+1}}$.
(d) Если $K+1\ge2u$, то $\big|\sum_{k>K}\tfrac{(-u)^k}{k\,k!}\big|\le 2\,\tfrac{u^{K+1}}{(K+1)(K+1)!}$.

*Доказательство.* (a) $E_1(u)=-\gamma-\ln u-\sum_{k\ge1}\frac{(-u)^k}{k\,k!}$ (A&S 5.1.11); подставьте в $e^{-u-E_1}$ и выделите член $k=1$ (он даёт $-u$). (b) $E_1>0$ даёт $F<e^{-u}$. Из (a): $-u-E_1(u)=\gamma+\ln u-u-\mathrm{Ein}(u)$, где $\mathrm{Ein}(u)=\int_0^u\frac{1-e^{-s}}{s}ds>0$, откуда $F<e^\gamma u$. (c) $e^uE_1(u)=\int_0^\infty\frac{e^{-s}}{u+s}ds$; подставьте $\frac1{u+s}=\sum_{j<K}\frac{(-s)^j}{u^{j+1}}+\frac{(-s)^K}{u^K(u+s)}$ и оцените остаток по $\frac1{u+s}\le\frac1u$. (d) Для $r_j=\frac{u^j}{j\,j!}$ имеем $\frac{r_{j+1}}{r_j}=\frac{u\,j}{(j+1)^2}\le\frac{u}{j+1}\le\frac12$ при $j\ge K+1$ — геометрическая оценка. $\square$

**Лемма 2 (подстановка и хвостовые тождества).** $u(t)$, $t\in\mathbb R$, — возрастающая биекция $\mathbb R\to(0,\infty)$ и $du=u(1+e^{-t})dt$, поэтому $\lambda=\int_{\mathbb R}G(t)\,dt$. Кроме того, для любого $t_0$
$$\int_{t_0}^{\infty}u e^{-u}(1+e^{-t})\,dt=e^{-u(t_0)},\qquad \int_{-\infty}^{t_0}e^\gamma u^2(1+e^{-t})\,dt=\tfrac12 e^\gamma u(t_0)^2 .$$
*(Прямая замена переменной $u=u(t)$.)* $\square$

**Лемма 3 (усечение суммы).** Пусть $h>0$, $u(K_Rh)\ge1$. Тогда
$$0\le\sum_{k>K_R}hG(kh)\le e^{-u(K_Rh)},\qquad 0\le\sum_{k<-K_L}hG(kh)\le\tfrac12e^\gamma u(-K_Lh)^2 .$$
*Доказательство.* По Лемме 1(b): $0<G\le g_R:=ue^{-u}(1+e^{-t})$ и $0<G\le g_L:=e^\gamma u^2(1+e^{-t})$. Функция $g_R$ убывает при $u\ge1$ (произведение двух убывающих положительных), поэтому $hg_R(kh)\le\int_{(k-1)h}^{kh}g_R$ и сумма по $k>K_R$ не больше $\int_{K_Rh}^\infty g_R=e^{-u(K_Rh)}$ (Лемма 2). Для $g_L$: $\frac{d}{dt}\ln g_L=2(1+e^{-t})-\frac{e^{-t}}{1+e^{-t}}>0$ — возрастает, $hg_L(kh)\le\int_{kh}^{(k+1)h}g_L$, сумма по $k\le-K_L-1$ не больше $\int_{-\infty}^{-K_Lh}g_L=\frac12e^\gamma u(-K_Lh)^2$. $\square$

**Лемма 4 (оценка в полосе).** Пусть $d=7/5$, $c=\cos d>0$. Для всех $w=t+iy$, $|y|\le d$ функция $G$ целая и
$$\int_{-\infty}^{\infty}|G(x+iy)|\,dx\le M:=M_1+M_2+M_3,$$
$$M_1=C_1e^{-2c}\big(\tfrac1c+\tfrac1{4c^2}\big),\quad C_1=e^{\gamma+2+(e-2)/2};\qquad M_2=4e^{\gamma}e^{4}\exp\!\big(2e^2+\tfrac12e^{e^2}\big);$$
$$M_3=\frac{e^{\,e^{e^{-2}-2}}\,(1+e^{-2})\,e^{-\kappa'e^2}}{\kappa'},\quad \kappa'=\cos(\theta_{\max})\,e^{-e^{-2}},\ \ \theta_{\max}=d+e^{-2}\sin d<\tfrac\pi2 .$$

*Доказательство.* $G$ целая, так как $F$ и $w\mapsto u(w)$ целые. Для $u=u(w)$: $|u|=\rho:=\exp(t-e^{-t}\cos y)$, $\arg u=\theta:=y+e^{-t}\sin y$ (это *действительная* мнимая часть $w-e^{-w}$), $|1+e^{-w}|\le1+e^{-t}$.

*Область $t\le0$.* $\rho\le e^{-c}<1$. По Лемме 1(a) $|F(u)|\le e^\gamma|u|\exp\!\big(2|u|+\sum_{k\ge2}\frac{|u|^k}{k\,k!}\big)\le C_1|u|$, так как $\sum_{k\ge2}\frac{1}{k\,k!}\le\frac{e-2}2$. Значит $|G|\le C_1\rho^2(1+e^{-t})\le C_1e^{2t-2ce^{-t}}(1+e^{-t})$. Замена $s=e^{-t}$ и $s^{-3}\le1$ при $s\ge1$ дают $\int_{-\infty}^0|G|\le C_1\int_1^\infty(1+s)e^{-2cs}ds=M_1$.

*Область $0\le t\le2$.* $\rho\le e^t\le e^2$ и $|F|\le e^\gamma\rho\exp(2\rho+\tfrac12e^{\rho})$, $1+e^{-t}\le2$, откуда $|G|\le B_2:=2e^{\gamma}e^4\exp(2e^2+\tfrac12e^{e^2})$, интеграл $\le2B_2=M_2$. (Оценка грубая, $M_2\approx10^{360}$; это безвредно, см. запас в §3.)

*Область $t\ge2$.* $|\theta|\le d+e^{-2}\sin d=\theta_{\max}<\pi/2$, поэтому $\operatorname{Re}u\ge\kappa\rho$, $\kappa=\cos\theta_{\max}>0$. В правой полуплоскости $E_1(u)=\int_0^\infty\frac{e^{-(u+r)}}{u+r}dr$ и $|u+r|\ge|u|$, значит $|E_1(u)|\le1/\rho\le e^{e^{-2}-2}$ (так как $\rho\ge e^{2-e^{-2}}$) и $|F|=e^{-\operatorname{Re}u-\operatorname{Re}E_1}\le\exp\big(e^{e^{-2}-2}\big)e^{-\kappa\rho}$. С $e^{t-e^{-2}}\le\rho\le e^t$ получаем $|G|\le e^{e^{e^{-2}-2}}(1+e^{-2})\,e^{t}e^{-\kappa'e^{t}}$, и $\int_2^\infty e^{t}e^{-\kappa'e^{t}}dt=e^{-\kappa'e^2}/\kappa'$. $\square$

*(Эти три оценки я дополнительно проверил численно на 4000 случайных точек полосы с `mpmath`: отношение $|G|$ к оценке нигде не превысило 0.52. Это проверка на опечатки, а не часть доказательства.)*

**Лемма 5 (теорема о трапециях).** (Trefethen–Weideman 2014, Thm 5.1.) Если $w$ аналитична в полосе $|\operatorname{Im}z|<a$, $w(z)\to0$ равномерно при $|\operatorname{Re}z|\to\infty$ и $\int|w(x+iy)|dx\le M$ при $|y|<a$, то
$$\Big|\int_{\mathbb R}w-h\sum_{k\in\mathbb Z}w(kh)\Big|\le\frac{2M}{e^{2\pi a/h}-1}.$$
Для $G$ условия выполнены с $a=d$ (затухание следует из оценок Леммы 4), поэтому ошибка дискретизации не превосходит $B_{\rm disc}=2M/(e^{2\pi d/h}-1)$.

**Лемма 6 (шары для узлов).** Для $k\in\mathbb Z$, $t=kh$: $G(t)=F(u)u(1+e^{-t})$ с $u=e^{t-e^{-t}}$.
* *Ряд* (любой $u>0$, но эффективен при $u\lesssim\mathrm{tol}/3$): по Лемме 1(a) $G=e^\gamma u^2(1+e^{-t})\exp(-2u+S_K+R)$, $|R|\le2\,\frac{u^{K+1}}{(K+1)(K+1)!}$ при $K+1\ge2u$ (Лемма 1(d)); всё считается шарами, рабочая точность повышена на $u/\ln2$ бит на сокращение членов. Сумма вычисляется *прямоугольным расщеплением*: блок из $m=8$ членов $a_{k_b+j}=a_{k_b}\,x^j\,\tfrac{k_b}{(k_b+j)\prod_{i=1}^{j}(k_b+i)}$ ($x=-u$) считается через степени $x^j$ и **малые целые** коэффициенты, на блок остаётся 2 полных умножения вместо $m$. Это точная перестановка конечной суммы (в шарах); остаток по-прежнему $2|a_{K+1}|$, а $a_{K+1}$ получается бесплатно как префактор следующего блока.
* *Цепная дробь* (средние и большие $u$): $e^{u}E_1(u)\in\big[f_{n-1},f_n\big]$ (Лемма 7), поэтому $E_1=e^{-u}\,(f_n+[-D,D])$, $D=|f_n-f_{n-1}|$. Подходящие дроби считаются прямой рекуррентной формулой, где шаги с $b_n=1$ требуют только малых множителей, а шаги с $b_n=u$ — одного полного умножения; рабочая точность ниже, чем у ряда (нужна лишь относительная точность $f$).
* *Асимптотика* (большие $u$): $E_1=e^{-u}\big(\sum_{j<K}\frac{(-1)^jj!}{u^{j+1}}+[-\tfrac{K!}{u^{K+1}},\tfrac{K!}{u^{K+1}}]\big)$ (Лемма 1(c)), $G=e^{-u-E_1}u(1+e^{-t})$.
* *Пренебрежимые узлы*: $0<G\le g_L$ или $g_R$ (Лемма 1(b)), узел заменяется шаром $[0,\,\text{оценка}]$.

В каждом случае после вычисления **проверяется** $\operatorname{rad}<\varepsilon=2^{-E}$; иначе точность поднимается и узел пересчитывается. Выбор метода и точности делается в `float`, но на *правильность* не влияет: обе формулы верны для любого $u>0$, а радиус проверяется строго.

**Лемма 7 (Стилтьес).** Для $z>0$ положим $A_{-1}=1,\ A_0=z,\ B_{-1}=0,\ B_0=1$ и
$$A_n=b_nA_{n-1}+a_nA_{n-2},\quad B_n=b_nB_{n-1}+a_nB_{n-2},\quad a_n=\lceil n/2\rceil,\ b_n=\begin{cases}1,&n\ \text{нечётное}\\ z,&n\ \text{чётное}\end{cases},\qquad f_n=\frac{B_n}{A_n}.$$
Тогда $e^{z}E_1(z)$ лежит между $f_{n-1}$ и $f_n$ при всех $n\ge1$.

*Обоснование.* $f_n=\tfrac1z g_n(1/z)$, где $g_n$ — подходящие дроби S-дроби $g(x)=\cfrac{1}{1+\cfrac{x}{1+\cfrac{x}{1+\cfrac{2x}{1+\cfrac{2x}{1+\cdots}}}}}$ с положительными коэффициентами $c_k=\lceil k/2\rceil$ для стилтьесовой функции $g(x)=\int_0^\infty\frac{e^{-s}}{1+sx}ds$ (ряд $\sum(-1)^kk!\,x^k$). Моменты $k!$ удовлетворяют условию Карлемана, поэтому проблема моментов определена, и по теореме Стилтьеса чётные подходящие дроби монотонно приближаются к $g(x)$ с одной стороны, нечётные — с другой, при всех $x>0$. Следовательно, $g$ лежит между соседними подходящими. Эквивалентность $f_n=\tfrac1zg_n(1/z)$ — обычное эквивалентное преобразование. (Численная проверка: 937 пар соседних подходящих дробей при $z\in[0.01,20000]$ — ни одного нарушения, знаки $f_n-f$ чередуются; это проверка на опечатки, не часть доказательства.) $\square$

**Теорема.** Пусть $S_{\rm fin}=h\sum_{k=-K_L}^{K_R}G(kh)$ (сумма шаров Леммы 6). Тогда
$$\lambda\in S_{\rm fin}+[0,\ \tau_R+\tau_L]+[-B_{\rm disc},B_{\rm disc}],\qquad \tau_R=e^{-u(K_Rh)},\ \tau_L=\tfrac12e^\gamma u(-K_Lh)^2 .$$
*Доказательство:* Лемма 2 даёт $\lambda=\int G$; Лемма 5: $\lambda=h\sum_{k\in\mathbb Z}G(kh)+\delta$, $|\delta|\le B_{\rm disc}$; Лемма 3: отбрасываемые хвосты суммы лежат в $[0,\tau_R]$ и $[0,\tau_L]$. $\square$

**Запас.** При $D=30000$ нужен $B_{\rm disc}<10^{-(D+G+6)}$. Строгая теорема даёт показатель $2\pi d\approx8.80$ вместо «эвристического» $\pi^2\approx9.87$, и даже с огромным $M\approx10^{360}$ запас остаётся (ячейка §3 проверяет это *в интервальной арифметике* и подбирает $h$).

## §2. Окружение и проверка API `python-flint`
Установка (нужен Internet = On на Kaggle) и проверка семантики методов, на которые опирается код.

In [ ]:
import math, os, sys, time, pickle, shutil, subprocess
import multiprocessing as _mp
from concurrent.futures import ProcessPoolExecutor, wait, FIRST_COMPLETED
try:
    import flint
except ImportError:                       # Kaggle: Settings -> Internet = On
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "python-flint"])
    import flint
from flint import arb, ctx

LN2 = math.log(2.0); LN10 = math.log(10.0); LOG2_10 = LN10 / LN2
EULER_F = 0.5772156649015329              # только для ОЦЕНОК точности (double); в доказательстве не участвует
print("python-flint", getattr(flint, "__version__", "?"), "| Python", sys.version.split()[0])

In [ ]:
def api_selftest():
    """Проверка семантики тех методов python-flint, на которые опирается доказательство."""
    ctx.prec = 256
    bad = []
    def chk(name, f):
        try:
            ok = bool(f())
        except Exception as ex:
            ok, name = False, "%s  [%s: %s]" % (name, type(ex).__name__, ex)
        if not ok:
            bad.append(name)
    third = arb(1) / 3
    chk("шар 1/3 имеет ненулевой радиус", lambda: third.rad() > 0)
    chk("lower()/upper() точные (радиус 0) и lower<upper",
        lambda: third.lower().rad() == 0 and third.upper().rad() == 0 and third.lower() < third.upper())
    def _bounds_ok():                      # при повышенной точности умножение на 3 точное, сравнения достоверны
        lo, up = third.lower(), third.upper()
        ctx.prec = 1024
        try:
            return 3 * lo <= 1 and 3 * up >= 1
        finally:
            ctx.prec = 256
    chk("lower <= 1/3 <= upper (через 3*x при точности 1024)", _bounds_ok)
    chk("floor().unique_fmpz()", lambda: int((arb(7)/2).floor().unique_fmpz()) == 3)
    chk("floor двузначного шара -> None", lambda: arb(3, 1).floor().unique_fmpz() is None)
    chk("arb(mid, rad)", lambda: arb(0, 1).upper() >= 1 and arb(0, 1).lower() <= -1)
    chk("степень", lambda: arb(3)**5 == 243)
    chk("сравнение с int", lambda: arb(5) > 4 and arb(5) < 6 and arb(5) >= 5 and arb(5) <= 5)
    chk("exp(1)", lambda: abs(arb(1).exp() - arb(2718281828459045235360287471352662497757) / arb(10)**39) < 1 / arb(10)**30)
    chk("pi", lambda: abs(arb.pi() - arb(3141592653589793238462643383279502884197) / arb(10)**39) < 1 / arb(10)**30)
    chk("gamma (const_euler)", lambda: abs(arb.const_euler() - arb(5772156649015328606065120900824024310421) / arb(10)**40) < 1 / arb(10)**38)
    chk("cos, sin", lambda: arb(0).cos() == 1 and arb(0).sin() == 0)
    ctx.prec = 600
    chk("лестница floor для 2^S-масштабирования", lambda: int(((arb(1)/3).lower() * arb(1 << 500)).floor().lower().unique_fmpz()) > 0)
    return bad

_bad = api_selftest()
if _bad:
    raise RuntimeError("API python-flint не соответствует ожиданиям:\n  - " + "\n  - ".join(_bad))
print("API-selftest: OK")

## §3. Константы, шаг $h$, хвосты и границы ошибок
`make_plan` вычисляет в шарах Arb: $M$ (Лемма 4), шаг $h$ (двоично-рациональный `double`, точное число) из условия $B_{\rm disc}<10^{-(D+G+6)}$, индексы $K_L,K_R$ и величины $\tau_L,\tau_R<10^{-(D+G+8)}$; все неравенства **проверяются** утверждениями `assert` в шарах. `to_fixed` переводит шар в пару целых $[L,H]$ (единица $2^{-S}$) с округлением наружу.

In [ ]:
def arb_ceil_int(x):
    """целое >= x (для шара x)"""
    return int(x.upper().floor().upper().unique_fmpz()) + 1

def to_fixed(x, S):
    """шар x -> целые (L, H) с  x ⊂ [L*2^-S, H*2^-S]  (строго; округление наружу)"""
    old = ctx.prec
    ctx.prec = max(old, S + 128)
    sc = arb(1 << S)
    L = int((x.lower() * sc).floor().lower().unique_fmpz())
    H = int((x.upper() * sc).floor().upper().unique_fmpz()) + 1
    ctx.prec = old
    return L, H

def strip_constants(prec=384):
    """Лемма 4: M >= sup_{|y|<=d} ∫|G(x+iy)|dx  при d = 7/5 (явные оценки трёх областей)."""
    ctx.prec = prec
    one = arb(1); pi = arb.pi(); gam = arb.const_euler(); e = one.exp()
    d = arb(7) / 5
    c = d.cos();  assert c > 0
    a = 2 * c
    # область t <= 0:  |u|<=1,  |F|<=C1|u|
    C1 = (gam + 2 + (e - 2) / 2).exp()
    M1 = C1 * (-a).exp() * (2 / a + 1 / (a * a))
    # область 0 <= t <= 2:  |u| <= e^2
    e2 = arb(2).exp(); em2 = (-arb(2)).exp()
    B2 = 2 * gam.exp() * (e2 * e2) * (2 * e2 + e2.exp() / 2).exp()
    M2 = 2 * B2
    # область t >= 2:  |arg u| <= theta_max < pi/2,  Re u >= kappa |u|
    th = d + em2 * d.sin();  assert th < pi / 2
    kap = th.cos();          assert kap > 0
    kp = kap * (-em2).exp()
    facE = ((em2 - 2).exp()).exp()          # exp(1/rho_min(2)) >= exp(|E1|)
    M3 = facE * (1 + em2) * (-kp * e2).exp() / kp
    return dict(d=d, M=M1 + M2 + M3, M1=M1, M2=M2, M3=M3, pi=pi, gam=gam)

def solve_t(target_lnu, lo, hi):
    """double: t с  t - e^{-t} = target_lnu  (монотонная функция)"""
    for _ in range(200):
        m = 0.5 * (lo + hi)
        if m - math.exp(-m) < target_lnu: lo = m
        else: hi = m
    return 0.5 * (lo + hi)

def make_plan(D, G=30):
    N = D + G
    K = strip_constants(); ctx.prec = 384
    M, d, pi, gam = K["M"], K["d"], K["pi"], K["gam"]
    ten = arb(10)
    # --- шаг h: из требования  2M/(e^{2 pi d/h}-1) < 10^-(N+6)
    L = (2 * M).log() + (N + 6) * ten.log()
    Lint = arb_ceil_int(L)
    h = math.floor(0.97 * 2 * math.pi * 1.4 / Lint * 2**40) / 2**40         # двоично-рациональный double
    hA = arb(h)
    Bdisc = 2 * M / ((2 * pi * d / hA).exp() - 1)
    assert Bdisc * ten**(N + 6) < 1, "оценка дискретизации не достигнута"
    # --- хвосты суммы
    UR = int(math.ceil((N + 8) * LN10)) + 2                                    # e^-UR < 10^-(N+8)
    tR = solve_t(math.log(UR), 0.0, 40.0);  K_R = int(math.ceil(tR / h)) + 1
    t = arb(K_R) * hA;  uR = (t - (-t).exp()).exp()
    assert uR >= UR and uR >= 1, "правый хвост"
    tailR = (-uR).exp()                                                        # Лемма 3: Σ_{k>K_R} h G <= e^{-u_R}
    lnuL = 0.5 * (-(N + 8) * LN10 + LN2 - EULER_F) - 1.0                       # e^g u_L^2/2 < 10^-(N+8)
    tL = solve_t(lnuL, -600.0, 0.0);  K_L = int(math.ceil(-tL / h)) + 1
    t = -(arb(K_L) * hA);  uL = (t - (-t).exp()).exp()
    tailL = gam.exp() * uL * uL / 2                                            # Лемма 3: Σ_{k<-K_L} h G <= e^g u_L^2/2
    assert tailL * ten**(N + 8) < 1, "левый хвост"
    Ebits = int(math.ceil((N + 8) * LOG2_10)) + 6                              # допуск на узел: eps = 2^-Ebits
    S = Ebits + 64                                                             # масштаб целочисленной арифметики
    pl = dict(D=D, G=G, N=N, h=h, K_L=K_L, K_R=K_R, Ebits=Ebits, S=S, EGbits=int(Ebits * 1.45) + 700, M=M, Bdisc=Bdisc, tailR=tailR, tailL=tailL,
              consts=K, nodes=K_L + K_R + 1)
    return pl

def log10_arb(x):
    """целая часть log10(x) (для печати)"""
    ctx.prec = 384
    return int((x.upper().log() / arb(10).log()).floor().lower().unique_fmpz())

## §4. Строгие шары для узлов $G(kh)$ (Лемма 6, 7)

**Откуда берётся скорость (и почему строгость не страдает).** Основная работа — узлы $1\lesssim u\lesssim\mathrm{tol}/3$: ряд требует $\approx6.5\,u$ членов, а каждый член — полное умножение на $u$ при ~$10^5$ бит; в сумме это порядка $10^9$ полных умножений. Поэтому:
1. *прямоугольное расщепление* ряда: в ~2.5 раза меньше полных умножений (проверено подсчётом операций);
2. *цепная дробь Стилтьеса* для больших $u$: число уровней падает с $\sim6.5u$ до $\sim\mathrm{X}\,(0.2+0.14\,\mathrm{X}/u)$, где $X=\ln(1/\text{допуск})-2u$, и считается она при меньшей точности;
3. *автотюн*: порог $u_{\rm cf}$, с которого цепная дробь быстрее ряда, определяется замером на этой машине при реальной точности (можно задать вручную).

**Выбор метода не влияет на строгость:** все три формулы верны для любого $u>0$, а радиус каждого узла проверяется (`rad < eps`) независимо от метода. Если метод не уложился, точность поднимается, а узел пересчитывается.

In [ ]:
ASYM_KMAX = 3000
RS_M = 8                        # размер блока прямоугольного расщепления
_EGC = {"bits": 0, "val": None}

def get_EG(bits):
    """e^gamma с точностью >= bits бит (кэш на процесс)"""
    if _EGC["val"] is None or _EGC["bits"] < bits:
        old = ctx.prec
        b = int(bits * 1.2) + 256
        ctx.prec = b
        _EGC["val"], _EGC["bits"] = arb.const_euler().exp(), b
        ctx.prec = old
    return _EGC["val"]

# ---------------------------------------------------------------- ряд (Лемма 1a, 1d)
def series_S_naive(u, uf, thr):
    """S = sum_{k=2}^K (-u)^k/(k k!) и rem: |sum_{k>K}| <= rem < thr (строго); K+1 >= 2u => хвост <= 2|a_{K+1}|.
       (эталонная медленная реализация: одно полное умножение на член)"""
    ub = u.upper()
    kmin = max(3, int(2 * uf * 1.01) + 6)
    while not (arb(kmin) >= 2 * ub):
        kmin += 8
    mu = -u
    p = mu
    S = arb(0)
    k = 1
    while True:
        k += 1
        p = p * mu / k
        S = S + p / k
        if k >= kmin and (k < 64 or k % 16 == 0):
            rem = 2 * abs(p).upper() * ub / ((k + 1) * (k + 1))
            if rem < thr:
                return S, rem

def _ln_a(k, xf):
    return k * xf - math.lgamma(k + 1.0) - math.log(k)

def _K_target(uf, xf, thr_bits):
    """(double) наименьшее K, при котором 2|a_{K+1}| < 2^-thr_bits и K+1 >= 2u; лишь для выбора числа блоков"""
    lim = -(thr_bits + 3) * LN2
    lo = max(3, int(2 * uf * 1.02) + 8)
    hi = lo
    while _ln_a(hi + 1, xf) > lim:
        hi = hi * 2
    while hi - lo > 1:
        mid = (lo + hi) // 2
        if _ln_a(mid + 1, xf) > lim: lo = mid
        else: hi = mid
    return hi

def series_S(u, uf, xf, thr_bits, m=None):
    """То же S и rem, но прямоугольным расщеплением: блок из m членов считается через степени x^j (j<=m) и
       МАЛЫЕ целые коэффициенты, на блок остаётся 2 полных умножения вместо m. Это точная перестановка
       конечной суммы S_K (в шарах), остаток - Лемма 1(d) по |a_{K+1}|, который получается бесплатно как
       префактор следующего блока."""
    m = m or RS_M
    thr = arb(1) / arb(1 << thr_bits)
    ub = u.upper()
    K = _K_target(uf, xf, thr_bits)
    if K < 4 * m:
        return series_S_naive(u, uf, thr)
    while True:
        nblocks = (K - 1 + m - 1) // m
        Kend = 1 + nblocks * m                                  # последний учтённый индекс
        if not (arb(Kend + 1) >= 2 * ub):                       # условие леммы 1(d): K+1 >= 2u
            K = int(K * 1.1) + 16
            continue
        x = -u
        xp = [arb(1), x]
        for j in range(2, m + 1):
            xp.append(xp[-1] * x)
        xm = xp[m]
        kb = 2
        pref = xp[2] / 4                                        # a_2 = x^2/(2*2!)
        S = arb(0)
        for _ in range(nblocks):
            inner = arb(1)                                      # j = 0: отношение 1
            Dprod = 1
            for j in range(1, m):                               # a_{kb+j}/a_{kb} = x^j * kb / ((kb+j) prod_{i=1..j}(kb+i))
                Dprod *= (kb + j)
                inner = inner + (xp[j] * kb) / ((kb + j) * Dprod)
            S = S + pref * inner
            Dprod *= (kb + m)
            pref = (pref * xm) * kb / ((kb + m) * Dprod)        # a_{kb+m}
            kb += m
        rem = 2 * abs(pref).upper()                             # pref = a_{Kend+1}
        if rem < thr:
            return S, rem
        K = int(K * 1.1) + 16                                   # (редко) не хватило - добавляем блоки

# ---------------------------------------------------------------- цепная дробь Стилтьеса (Лемма 7)
class _Escalate(Exception):
    pass

def cf_f_hull(u, uf, gap_bits, nmax):
    """Шар для f = e^u E1(u), u>0:  f = B_n/A_n  (подходящие дроби цепной дроби 1/(z+1/(1+1/(z+2/(1+2/(z+...)))))).
       По теореме Стилтьеса f лежит между f_{n-1} и f_n, поэтому |f - f_n| <= |f_n - f_{n-1}|."""
    thr = arb(1) / arb(1 << gap_bits)
    A2, A1 = arb(1), u                 # A_{-1}, A_0
    B2, B1 = arb(0), arb(1)
    n = 0
    while n < nmax:
        for _ in range(32):
            n += 1
            a = (n + 1) // 2
            if n & 1:                  # b_n = 1: только малые множители
                A2, A1 = A1, A1 + a * A2
                B2, B1 = B1, B1 + a * B2
            else:                      # b_n = u: одно полное умножение на последовательность
                A2, A1 = A1, u * A1 + a * A2
                B2, B1 = B1, u * B1 + a * B2
        fn = B1 / A1
        D = abs(fn - B2 / A2).upper()
        if D < thr:
            return fn + arb(0, D)
    raise _Escalate("цепная дробь: превышен nmax")

# ---------------------------------------------------------------- асимптотика (Лемма 1c)
def asym_K(xf, uf, lng, L_eps):
    """минимальное K (double - только выбор метода), при котором остаток асимптотического ряда достаточно мал"""
    top = min(ASYM_KMAX, int(uf) - 1)
    lnfact = 0.0
    for K in range(0, top + 1):
        if K > 0:
            lnfact += math.log(K)
        if lng + lnfact - (K + 1) * xf - uf <= -L_eps - 3.0:
            return K
    return None

def _mk_u(k, pl):
    t = arb(pl["h"]) * k
    em = (-t).exp()
    return em, (t - em).exp()

def _eval_series(k, pl, P, thr_bits, uf, xf):
    Pv = P + (int(uf / LN2) + 16 if uf > 1 else 0)          # запас на сокращение членов (~e^u)
    ctx.prec = Pv
    em, u = _mk_u(k, pl)
    S, rem = series_S(u, uf, xf, thr_bits)
    expo = -2 * u + S + arb(0, rem)                         # F = e^g u exp(-2u + S)        (Лемма 1a)
    return get_EG(Pv) * u * u * (1 + em) * expo.exp()       # G = F u (1 + e^-t)

def _eval_cf(k, pl, P, gap_bits, uf, nmax):
    ctx.prec = P
    em, u = _mk_u(k, pl)
    ctx.prec = gap_bits + 96 + int(math.log2(nmax + 2))     # f нужна с ОТНОСИТЕЛЬНОЙ точностью ~2^-gap_bits (дешевле, чем P)
    f = cf_f_hull(u, uf, gap_bits, nmax)
    ctx.prec = P
    E1 = (-u).exp() * f                                      # E1 = e^-u f
    return (-u - E1).exp() * u * (1 + em)

def _eval_asym(k, pl, P, K):
    ctx.prec = P
    em, u = _mk_u(k, pl)
    w = (-u).exp()
    if K == 0:
        Ssum = arb(0)
    else:
        Ssum = arb(1)
        for j in range(K - 1, 0, -1):                       # Горнер: sum_{j<K} (-1)^j j!/u^j
            Ssum = 1 - Ssum * j / u
    f = arb(1)
    for j in range(1, K + 1):
        f = f * j / u
    rem = f / u                                              # K!/u^(K+1)       (Лемма 1c, остаток)
    E1 = w * (Ssum / u + arb(0, rem))                        # E1(u) = e^-u (S/u + R),  |R| <= K!/u^(K+1)
    return (-u - E1).exp() * u * (1 + em)

def _est(k, pl):
    h = pl["h"]
    tf = k * h
    emf = math.exp(-tf); xf = tf - emf
    uf = math.exp(xf) if xf < 700.0 else math.inf
    l1p = math.log1p(emf)
    lng = (EULER_F + 2 * xf + l1p) if xf < 0 else (xf - uf + l1p)     # ln(верхней оценки G)
    return xf, uf, lng

def method_of(k, pl):
    """(только для статистики/автотюна) какой метод выберет node_ball"""
    xf, uf, lng = _est(k, pl)
    L_eps = pl["Ebits"] * LN2
    if L_eps + lng + 2.0 < 0: return "skip"
    if xf >= 0 and asym_K(xf, uf, lng, L_eps) is not None: return "asym"
    if xf >= 0 and uf >= pl.get("cf_min_u", 1e18): return "cf"
    return "series"

def node_ball(k, pl, force=None):
    """строгий шар для G(kh) = F(u) u (1+e^-t), u = exp(t - e^-t), t = kh; радиус < eps = 2^-Ebits"""
    Eb = pl["Ebits"]
    xf, uf, lng = _est(k, pl)
    L_eps = Eb * LN2
    eps = arb(1) / arb(1 << Eb)
    need_ln = L_eps + lng + 2.0
    if need_ln < 0:                                                   # узел < eps: шар [0, оценка]
        ctx.prec = 128
        em, u = _mk_u(k, pl)
        g = (get_EG(128) * u * u * (1 + em)) if xf < 0 else (u * (-u).exp() * (1 + em))
        ub = g.upper()
        return ub / 2 + arb(0, ub / 2)                                # шар [0, ub]
    P = max(64, int(need_ln / LN2) + int(math.log2(uf + 2.0)) + 64)
    thr_bits = int(need_ln / LN2) + 4
    gap_bits = max(32, int((L_eps + lng - uf + 4.0) / LN2) + 2) if xf >= 0 else 0
    meth = force or method_of(k, pl)
    Ka = asym_K(xf, uf, lng, L_eps) if (xf >= 0 and meth == "asym") else None
    X = max(L_eps + lng - uf + xf, 1.0) if xf >= 0 else 1.0           # нужная точность f, нат (оценка числа уровней)
    nmax = int(3.0 * X * (0.22 + 0.14 * X / max(uf, 1.0))) + 4000
    for attempt in range(8):
        try:
            if meth == "asym" and Ka is not None:
                g = _eval_asym(k, pl, P, Ka)
            elif meth == "cf":
                g = _eval_cf(k, pl, P, gap_bits, uf, nmax)
            else:
                g = _eval_series(k, pl, P, thr_bits, uf, xf)
            if g.is_finite() and g.rad() < eps:
                return g
        except _Escalate:
            nmax *= 2
        P += max(64, P // 4); thr_bits += max(64, thr_bits // 4); gap_bits += max(32, gap_bits // 4)
    raise ArithmeticError("узел k=%d: радиус не достигнут" % k)

def autotune_cf_min_u(pl, log=print):
    """Подбор порога u, с которого цепная дробь быстрее ряда (на ЭТОЙ машине и при ЭТОЙ точности).
       Корректность не зависит от порога: оба метода строгие."""
    L_eps = pl["Ebits"] * LN2
    cand = None
    for uu in (20000, 16000, 13000, 10000, 8000, 6000, 4500, 3000):
        if uu > 0.30 * L_eps:                      # выше - работает асимптотика / нет таких узлов
            continue
        t = solve_t(math.log(uu), 0.0, 40.0)
        k = int(round(t / pl["h"]))
        res = {}
        for name in ("cf", "series"):
            t0 = time.time()
            node_ball(k, pl, force=name)
            res[name] = time.time() - t0
        log("  автотюн u~%6d: ряд %.2f с | цепная дробь %.2f с" % (uu, res["series"], res["cf"]))
        if res["cf"] < 0.9 * res["series"]: cand = uu
        else: break
    return float(cand) if cand is not None else 1e18

## §5. Параллельный расчёт с чекпоинтами
Куски узлов считаются в процессах; результат куска — пара целых $[L,H]$ (строгое включение суммы шаров), поэтому через границы процессов ничего не теряется. Чекпоинты (`pickle`) позволяют продолжить после обрыва сессии.

In [ ]:
def run_chunk(lo, hi, pl):
    S = pl["S"]
    get_EG(pl["EGbits"])                       # e^gamma один раз на процесс, с запасом точности
    ctx.prec = pl["Ebits"] + 256
    se, so = arb(0), arb(0)
    for k in range(lo, hi):
        g = node_ball(k, pl)
        ctx.prec = pl["Ebits"] + 256
        if k & 1: so = so + g
        else:     se = se + g
    return lo, hi, to_fixed(se, S), to_fixed(so, S)

def compute_chunks(pl, jobs, chunk, ckpt, deadline=None, log=print):
    pw = {k: pl[k] for k in ("D", "G", "h", "Ebits", "S", "EGbits", "K_L", "K_R", "cf_min_u")}   # в процессы передаём только числа
    n0, n1 = -pl["K_L"], pl["K_R"] + 1
    bounds = [(lo, min(lo + chunk, n1)) for lo in range(n0, n1, chunk)]
    meta = (pl["D"], pl["G"], pl["h"], pl["Ebits"], pl["K_L"], pl["K_R"], chunk, "v2")
    done = {}
    if ckpt:
        os.makedirs(ckpt, exist_ok=True)
        mp = os.path.join(ckpt, "meta.pkl")
        if os.path.exists(mp):
            with open(mp, "rb") as f:
                if pickle.load(f) != meta:
                    raise RuntimeError(ckpt + ": чекпоинты другого прогона - удалите папку или смените ckpt")
        else:
            with open(mp, "wb") as f: pickle.dump(meta, f)
        for lo, hi in bounds:
            p = os.path.join(ckpt, "c_%d.pkl" % lo)
            if os.path.exists(p):
                try:
                    with open(p, "rb") as f: c = pickle.load(f)
                    if c[0] == lo and c[1] == hi: done[lo] = c
                except Exception:
                    pass
    todo = sorted([b for b in bounds if b[0] not in done], key=lambda b: -b[0])      # дорогие (большие u) - первыми
    if done: log("  из чекпоинта: %d/%d кусков" % (len(done), len(bounds)))
    t0 = time.time(); fin = 0
    def accept(c):
        nonlocal fin
        done[c[0]] = c
        if ckpt:
            p = os.path.join(ckpt, "c_%d.pkl" % c[0])
            with open(p + ".tmp", "wb") as f: pickle.dump(c, f)
            os.replace(p + ".tmp", p)
        fin += 1
        el = time.time() - t0
        print("\r  куски %d/%d  прошло %6.0fс  осталось ~%6.0fс   " % (len(done), len(bounds), el, el / fin * (len(todo) - fin)),
              end="", flush=True)
    stopped = False
    if jobs <= 1 or len(todo) <= 1:
        for lo, hi in todo:
            if deadline and time.time() > deadline: stopped = True; break
            accept(run_chunk(lo, hi, pw))
    else:
        with ProcessPoolExecutor(max_workers=jobs, mp_context=_mp.get_context("fork")) as ex:
            pending = {ex.submit(run_chunk, lo, hi, pw) for lo, hi in todo}
            while pending:
                tmo = None if not deadline else max(1.0, min(30.0, deadline - time.time()))
                fin_set, pending = wait(pending, timeout=tmo, return_when=FIRST_COMPLETED)
                for f in fin_set: accept(f.result())
                if deadline and time.time() > deadline and pending:
                    stopped = True
                    for f in pending: f.cancel()
                    for f in [f for f in pending if not f.cancelled()]: accept(f.result())
                    pending = set()
    if todo: print()
    if stopped: return None
    return [done[k] for k in sorted(done)]

## §6. Итоговый сертификат
Всё в точной целочисленной арифметике Python: $\Lambda_-=\lfloor hL\rfloor-B$, $\Lambda_+=\lceil hH\rceil+\tau_R+\tau_L+B$ (в единицах $2^{-S}$), затем $\lfloor10^D\Lambda_\pm 2^{-S}\rfloor$.

In [ ]:
def certify(pl, chunks, verbose=True):
    """Итоговый строгий интервал для lambda в целых числах (единица = 2^-S) и сертификат цифр."""
    S, D = pl["S"], pl["D"]
    hn, hd = pl["h"].as_integer_ratio()                       # h = hn/hd точно (hd - степень двойки)
    Lsum = sum(c[2][0] + c[3][0] for c in chunks)
    Hsum = sum(c[2][1] + c[3][1] for c in chunks)
    lo_fin = (hn * Lsum) // hd                                # пол
    hi_fin = -((-hn * Hsum) // hd)                            # потолок
    B = to_fixed(pl["Bdisc"], S)[1]                           # |ошибка дискретизации| <= B (Лемма 5)
    TR = to_fixed(pl["tailR"], S)[1]; TL = to_fixed(pl["tailL"], S)[1]     # хвосты >= 0 (Лемма 3)
    lam_lo = lo_fin - B
    lam_hi = hi_fin + TR + TL + B
    assert lam_lo > 0 and lam_hi > lam_lo
    scale = 10 ** D
    a = (lam_lo * scale) >> S; b = (lam_hi * scale) >> S
    res = dict(lam_lo=lam_lo, lam_hi=lam_hi, S=S, proven=(a == b), digits=(str(a).zfill(D) if a == b else None),
               digits_lo=str(a).zfill(D), digits_hi=str(b).zfill(D))
    if verbose:
        w = lam_hi - lam_lo
        print("ширина строгого интервала ~ 2^-%d ≈ 10^-%.1f" % (S - w.bit_length(), (S - w.bit_length()) * math.log10(2)))
        print("оценка дискретизации B ≈ 10^%d,  хвосты ≈ 10^%d / 10^%d" % (log10_arb(pl["Bdisc"]), log10_arb(pl["tailR"]), log10_arb(pl["tailL"])))
        print("ВСЕ %d ЗНАКОВ ДОКАЗАНЫ (floor(lo*10^D) == floor(hi*10^D))" % D if res["proven"] else
              "ЗНАКИ НЕ ДОКАЗАНЫ: нижняя и верхняя границы различаются - увеличьте GUARD")
    return res

In [ ]:
KNOWN = ("6243299885435508709929363831008372441796426201805292869735519024"
         "956380888551132544624602761955398688")        # OEIS A084945: первые 98 знаков после запятой (для независимой сверки)

def run_certificate(D, G=30, jobs=0, chunk=0, ckpt=None, out=None, max_hours=0.0, cf_min_u=None):
    t0 = time.time()
    try: ncpu = len(os.sched_getaffinity(0))
    except AttributeError: ncpu = os.cpu_count() or 1
    jobs = jobs if jobs > 0 else (min(ncpu, 4) if os.path.isdir("/kaggle/working") else ncpu)
    base = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
    if ckpt is None: ckpt = os.path.join(base, "gd_rigorous_ckpt_%d" % D)
    pl = make_plan(D, G)
    print("D=%d  GUARD=%d  узлов=%d  h=%.6e  K_L=%d K_R=%d  процессов=%d" % (D, G, pl["nodes"], pl["h"], pl["K_L"], pl["K_R"], jobs))
    print("  M ≈ 10^%d,  B_disc ≈ 10^%d" % (log10_arb(pl["M"]), log10_arb(pl["Bdisc"])))
    if cf_min_u is None:
        cf_min_u = autotune_cf_min_u(pl)
    pl["cf_min_u"] = float(cf_min_u)
    print("  порог цепной дроби: u >= %s" % ("∞ (только ряд)" if cf_min_u > 1e17 else "%.0f" % cf_min_u))
    if chunk <= 0: chunk = max(25, min(200, pl["nodes"] // (max(1, jobs) * 24)))
    deadline = time.time() + max_hours * 3600.0 if max_hours > 0 else None
    chunks = compute_chunks(pl, jobs, chunk, ckpt, deadline)
    if chunks is None:
        print("ОСТАНОВЛЕНО по max_hours: чекпоинты в", ckpt, "- запустите ячейку снова"); return None
    res = certify(pl, chunks)
    k = min(D, 95)
    ok_known = res["digits_lo"][:k] == KNOWN[:k]
    print("независимая сверка с OEIS A084945 (первые %d знаков): %s" % (k, "OK" if ok_known else "НЕ СОВПАЛО"))
    if res["proven"] and ok_known:
        path = out if out is not None else os.path.join(base, "golomb_dickman_%d_certified.txt" % D)
        with open(path, "w") as f:
            f.write("# Golomb-Dickman constant, %d decimals, CERTIFIED by interval arithmetic (trapezoid rule + Arb balls)\n" % D)
            f.write("# S=%d lam_lo=%d\n# lam_hi=%d\n" % (res["S"], res["lam_lo"], res["lam_hi"]))
            f.write("0." + res["digits"] + "\n")
        print("записано", path, "| время %.0f с" % (time.time() - t0))
    return res

## §7. Запуск
Сначала **проба на 60 знаках** (секунды): она проходит весь конвейер, включая сверку с OEIS, и останавливает ноутбук, если что-то не так. Затем основной прогон.

In [ ]:
smoke = run_certificate(60, G=30, jobs=0, ckpt=None, out="smoke_60.txt", cf_min_u=20.0)   # cf_min_u=20: проба проходит ВСЕ пути (ряд, цепная дробь, асимптотика)
assert smoke and smoke["proven"], "проба не прошла - основной прогон не запускаем"
print("проба OK")

In [ ]:
# ===== ОСНОВНОЙ ПРОГОН =====  (часы; можно прерывать - чекпоинты сохраняются)
DIGITS    = 30000
GUARD     = 30        # запасные знаки; сертификат требует, чтобы знаки DIGITS+1..DIGITS+GUARD не были цепочкой 0/9
MAX_HOURS = 8.5       # мягкая остановка перед лимитом сессии (0 = без лимита)
CF_MIN_U  = None      # None = автотюн (около минуты); число = порог u для цепной дроби (например, 13000.0); 1e18 = только ряд
RESULT = run_certificate(DIGITS, G=GUARD, jobs=0, max_hours=MAX_HOURS, cf_min_u=CF_MIN_U)

## Что делать дальше и честные оговорки

* **Если сработал `MAX_HOURS`**: сохраните версию ноутбука (*Save Version*) — чекпоинты лежат в `/kaggle/working/gd_rigorous_ckpt_30000`. В новой сессии добавьте этот Output как Input, скопируйте папку обратно в `/kaggle/working/` и запустите ячейку основного прогона снова.
* **Если «ЗНАКИ НЕ ДОКАЗАНЫ»**: интервал содержит границу между двумя десятичными ячейками; поднимите `GUARD` (например, до 60) и запустите заново (чекпоинты другого `GUARD` не подходят — смените/удалите папку).
* **Объём работы.** Основные затраты — узлы $1\lesssim u\lesssim\mathrm{tol}/3$. По подсчёту операций новая версия делает примерно в 2.5 раза меньше полных умножений на узлах с рядом и на порядок меньше на узлах с большими $u$ (цепная дробь), так что ожидаю общее ускорение в 3–5 раз по сравнению с чистым рядом. **Это оценка по структуре алгоритма, а не измерение на реальном Arb**: время на вашей машине покажет автотюн (он печатает времена ряда и цепной дроби) и пробные запуски `DIGITS=1000`, `3000` (рост примерно как $D^{2.5\ldots3}$).
* **Проверка этого ноутбука.** На стороне автора он исполнялся **не на настоящем Arb**, а на имитации шаров поверх `mpmath` (все ветви узлов, чекпоинты, параллельность): при $D=60$ и $D=150$ сертификат проходит и строгий интервал содержит независимо вычисленное `mpmath.quad`. Работу с реальным `python-flint 0.9` проверяет `selftest` в первой ячейке — если он что-то отвергнет, ноутбук остановится с понятным сообщением.
* **Чего это не даёт.** Доказательство условно по отношению к базе доверия из начала ноутбука (корректность Arb и цитируемая теорема Trefethen–Weideman).